## Read the Trained Weight and Run FP32 Inference

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
###########################################################################
# Dataset
###########################################################################

data = {
    'Name': [
        'Nasi Tim Ayam',
        'Telur Balado',
        'Sayur Bening Bayam',
        'Ayam Goreng Mentega',
        'Tumis Buncis Wortel',
        'Opor Ayam'
    ],
    'Sour': [1, 2, 2, 2, 1, 1],
    'Sweet': [6, 4, 3, 8, 4, 5],
    'Salty': [5, 6, 4, 6, 4, 5],
    'Spicy': [1, 7, 1, 2, 2, 2],
    # Target: 1 for Like, 0 for Dislike
    'Label': [1, 0, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

# Features (Sour, Sweet, Salty, Spicy) -> 4 input dimensions
X = df[['Sour', 'Sweet', 'Salty', 'Spicy']].values
y = df['Label'].values

# Standardize feature values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Shape: (N, 1)

In [3]:
###########################################################################
# Re-define the model architecture (must match the saved model)
###########################################################################

class FoodPreferenceANN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8):
        super(FoodPreferenceANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

In [4]:
###########################################################################
# Load trained weights from .pt File
###########################################################################

model = FoodPreferenceANN(input_dim=4, hidden_dim=8)
model.load_state_dict(torch.load('ai_food_weights.pt'))
model.eval()  # Set model to evaluation mode

print("Model weights successfully loaded from ai_food_weights.pt!")

Model weights successfully loaded from ai_food_weights.pt!


In [5]:
###########################################################################
# Perform inference using the original FP32 model
###########################################################################

model.eval()

with torch.no_grad():
    raw_logits_fp32 = model(X_tensor)
    probabilities_fp32 = torch.sigmoid(raw_logits_fp32)
    predictions_fp32 = (probabilities_fp32 >= 0.5).float()

print("\n--- FP32 Model Predictions ---")

for name, prob, pred in zip(
    df['Name'],
    probabilities_fp32,
    predictions_fp32
):
    taste = "Like" if pred.item() else "Dislike"

    print(
        f"Food: {name:22s} | "
        f"Prob(Like): {prob.item():.4f} | "
        f"Predicted: {taste}"
    )

# --- Calculate Accuracy ---
correct = (predictions_fp32 == y_tensor).sum().item()
total = y_tensor.size(0)
accuracy = correct / total * 100

print(f"\nFP32 Model Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")


--- FP32 Model Predictions ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9892 | Predicted: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Predicted: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Predicted: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9961 | Predicted: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0387 | Predicted: Dislike
Food: Opor Ayam              | Prob(Like): 0.9408 | Predicted: Like

FP32 Model Accuracy: 100.00% (6/6 correct)


## Quantization and Run INT8 Inference

In [6]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import copy

This function implements a pure Python reference implementation of 2D matrix multiplication designed to simulate hardware execution on an AI accelerator or FPGA. It reshaping the input arrays, validates matching inner dimensions, and uses explicit nested loops to compute dot products.

In [7]:
###########################################################################
# Naive matrix multiplication
###########################################################################

def manual_mmult(A, B):
    # Ensure inputs are 2D
    A_2d = A if A.ndim == 2 else A.reshape(-1, A.shape[-1])
    B_2d = B if B.ndim == 2 else B.reshape(-1, B.shape[-1])

    rows_A, cols_A = A_2d.shape
    rows_B, cols_B = B_2d.shape

    if cols_A != rows_B:
        raise ValueError(
            f"Incompatible shapes for matmul: {cols_A} != {rows_B}"
        )

    # Initialize 2D output array (using int32 for FPGA accumulation)
    result_2d = np.zeros((rows_A, cols_B), dtype=np.int32)

    # Standard matrix multiplication loops: (M x K) @ (K x N) -> (M x N)
    for i in range(rows_A):
        for j in range(cols_B):
            acc = 0
            for k in range(cols_A):
                acc += int(A_2d[i, k]) * int(B_2d[k, j])
            result_2d[i, j] = acc

    return result_2d

This function calculates the linear quantization scale factors for both the activations and weights across every layer of the trained model. It maps these maximum values, along with the peak weight values from each linear layer, into the symmetric 8-bit signed integer range (-127 to 127) and returns the resulting scales in a dictionary.

In [8]:
###########################################################################
# Function to get quantization scales
###########################################################################

def get_linear_quantization_scales(model, calibration_loader):
    model.eval()

    # Track activation ranges entering each Linear layer
    max_input_layer1 = 0.0
    max_input_layer2 = 0.0

    with torch.no_grad():
        for batch in calibration_loader:

            # Unpack input tensor from DataLoader
            x = batch[0] if isinstance(batch, (list, tuple)) else batch

            # ---------------------------------------------------------
            # Input to Linear 1
            # ---------------------------------------------------------
            max_input_layer1 = max(
                max_input_layer1,
                torch.max(torch.abs(x)).item()
            )

            # Floating-point Linear 1
            layer1_out = model.layer1(x)

            # Floating-point ReLU
            relu_out = model.relu(layer1_out)

            # ---------------------------------------------------------
            # Input to Linear 2
            # ---------------------------------------------------------
            max_input_layer2 = max(
                max_input_layer2,
                torch.max(torch.abs(relu_out)).item()
            )

    # ---------------------------------------------------------
    # Activation quantization scales
    # ---------------------------------------------------------
    input_scale_layer1 = max_input_layer1 / 127.0
    input_scale_layer2 = max_input_layer2 / 127.0

    # ---------------------------------------------------------
    # Weight quantization scales
    # ---------------------------------------------------------
    weight_scale_layer1 = (
        torch.max(torch.abs(model.layer1.weight)).item() / 127.0
    )

    weight_scale_layer2 = (
        torch.max(torch.abs(model.layer2.weight)).item() / 127.0
    )

    return {
        "layer1_input_scale": input_scale_layer1,
        "layer1_weight_scale": weight_scale_layer1,
        "layer2_input_scale": input_scale_layer2,
        "layer2_weight_scale": weight_scale_layer2,
    }

This code prepares a calibration pipeline by wrapping the input feature tensor into a PyTorch TensorDataset and feeding it through a DataLoader with a batch size of 2. It then passes the model and calibration loader into get_linear_quantization_scales() to compute the required INT8 scale factors for each layer, printing the resulting dictionary of values.

In [9]:
###########################################################################
# Get quantization scale
###########################################################################

# Calibration dataset uses only the input features
calibration_dataset = TensorDataset(X_tensor)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=2,
    shuffle=False
)

# Calculate INT8 activation and weight scales
scales = get_linear_quantization_scales(
    model,
    calibration_loader
)

print(scales)

{'layer1_input_scale': 0.01718756345313365, 'layer1_weight_scale': 0.009734624952781858, 'layer2_input_scale': 0.031243701619426096, 'layer2_weight_scale': 0.007930493730259693}


This PyTorch module defines QuantizedLinear, a custom linear layer that replaces standard FP32 dense operations with simulated INT8 integer arithmetic. During initialization, it converts and clips the floating-point weights into signed INT8 integers using a provided scale factor while preserving bias in FP32. During the forward pass, floating-point inputs are quantized to INT8, multiplied against the integer weights using manual_mmult to yield an INT32 accumulated result, and then rescaled back into floating-point representation before adding bias.

In [10]:
###########################################################################
# Quantized INT8 linear class
###########################################################################

class QuantizedLinear(nn.Module):

    def __init__(self, original_layer, input_scale, weight_scale):
        super().__init__()

        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        # Quantization scales
        self.input_scale = input_scale
        self.weight_scale = weight_scale

        # ---------------------------------------------------------------
        # Quantize weights using the calibrated weight scale
        # ---------------------------------------------------------------
        weights_float = original_layer.weight.detach().cpu().numpy()

        self.weights_int8 = np.round(
            weights_float / self.weight_scale
        ).clip(-128, 127).astype(np.int8)

        # Keep bias in floating point for simplicity
        if original_layer.bias is not None:
            self.bias = original_layer.bias.detach().clone()
        else:
            self.bias = None

    def forward(self, x):

        # FP32 -> INT8
        x_np = x.detach().cpu().numpy()

        x_int8 = np.round(
            x_np / self.input_scale
        ).clip(-128, 127).astype(np.int8)

        # INT8 x INT8 -> INT32
        x1_shape = x_int8.shape
        x2_shape = self.weights_int8.T.shape
        print(f"DEBUG: matmul X1{x1_shape} * X2{x2_shape}")

        out_int32 = manual_mmult(
            x_int8.astype(np.int32),
            self.weights_int8.T.astype(np.int32)
        )

        # INT32 -> FP32
        total_scale = self.input_scale * self.weight_scale

        out_float = (
            out_int32.astype(np.float32) * total_scale
        )

        # Add FP32 bias
        if self.bias is not None:
            out_float += self.bias.cpu().numpy()

        return torch.from_numpy(out_float).to(x.device)

This function traverses a PyTorch model's module tree and replaces all standard nn.Linear layers with custom QuantizedLinear modules using the calibrated input and weight scales. It identifies target layers by name, maps their specific quantization scale factors, instantiates the quantized counterparts, and updates the model in-place while logging the replacement details.

In [11]:
###########################################################################
# Function to integrate quantized linear class
###########################################################################

def integrate_quantization_layer(model, scales):
    """
    Recursively replaces all nn.Linear layers with NumpyQuantizedLinear.

    Each Linear layer uses its corresponding calibrated input
    and weight quantization scales.
    """

    for name, module in model.named_children():

        if isinstance(module, nn.Linear):

            # ---------------------------------------------------------
            # Select quantization scales for this layer
            # ---------------------------------------------------------
            if name == "layer1":
                input_scale = scales["layer1_input_scale"]
                weight_scale = scales["layer1_weight_scale"]

            elif name == "layer2":
                input_scale = scales["layer2_input_scale"]
                weight_scale = scales["layer2_weight_scale"]

            else:
                raise ValueError(
                    f"No quantization scale found for layer: {name}"
                )

            # ---------------------------------------------------------
            # Create quantized layer
            # ---------------------------------------------------------
            quantized_layer = QuantizedLinear(
                module,
                input_scale,
                weight_scale
            )

            # ---------------------------------------------------------
            # Replace FP32 Linear with quantized Linear
            # ---------------------------------------------------------
            setattr(model, name, quantized_layer)

            print(
                f"Replaced layer: {name} "
                f"with NumpyQuantizedLinear "
                f"(input_scale={input_scale:.6f}, "
                f"weight_scale={weight_scale:.6f})"
            )

        else:
            # Recursively process child modules
            integrate_quantization_layer(module, scales)

This code creates a deep copy of the original floating-point model to preserve the baseline FP32 weights without modification. It then calls integrate_quantization_layer() to replace the standard linear layers of the copied model with custom INT8 quantized layers, printing a side-by-side comparison of the model architectures before and after conversion.

In [12]:
###########################################################################
# Replace the FP32 linear class with INT8 linear class
###########################################################################

# Create an isolated deep copy for quantization
# The original FP32 model remains unchanged.
model_quantized = copy.deepcopy(model)

print("\n--- Before Quantization ---")
print("Original model:")
print("  Layer 1:", model.layer1)
print("  Layer 2:", model.layer2)

print("\nQuantized model copy:")
print("  Layer 1:", model_quantized.layer1)
print("  Layer 2:", model_quantized.layer2)
print("")

# Apply quantization ONLY to the copied model
integrate_quantization_layer(model_quantized, scales)

print("\n--- After Quantization ---")
print("Original model (unchanged):")
print("  Layer 1:", model.layer1)
print("  Layer 2:", model.layer2)

print("\nQuantized model:")
print("  Layer 1:", model_quantized.layer1)
print("  Layer 2:", model_quantized.layer2)


--- Before Quantization ---
Original model:
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Quantized model copy:
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Replaced layer: layer1 with NumpyQuantizedLinear (input_scale=0.017188, weight_scale=0.009735)
Replaced layer: layer2 with NumpyQuantizedLinear (input_scale=0.031244, weight_scale=0.007930)

--- After Quantization ---
Original model (unchanged):
  Layer 1: Linear(in_features=4, out_features=8, bias=True)
  Layer 2: Linear(in_features=8, out_features=1, bias=True)

Quantized model:
  Layer 1: QuantizedLinear()
  Layer 2: QuantizedLinear()


This code evaluates the fully quantized INT8 model on the dataset. It passes the feature tensor through the modified network to produce quantized logits, applies a sigmoid function to compute output probabilities, and maps them to binary "Like" or "Dislike" predictions based on a 0.5 threshold. Finally, it prints the prediction results for each dish alongside the overall accuracy achieved by the simulated low-precision hardware pipeline.

In [13]:
###########################################################################
# Perform inference using the INT8 quantized model
###########################################################################

model_quantized.eval()

with torch.no_grad():
    raw_logits_int8 = model_quantized(X_tensor)
    probabilities_int8 = torch.sigmoid(raw_logits_int8)
    predictions_int8 = (probabilities_int8 >= 0.5).float()

print("\n--- INT8 Model Predictions ---")

for name, prob, pred in zip(
    df['Name'],
    probabilities_int8,
    predictions_int8
):
    taste = "Like" if pred.item() else "Dislike"

    print(
        f"Food: {name:22s} | "
        f"Prob(Like): {prob.item():.4f} | "
        f"Predicted: {taste}"
    )

# --- Calculate Accuracy ---
correct = (predictions_int8 == y_tensor).sum().item()
total = y_tensor.size(0)
accuracy = correct / total * 100

print(
    f"\nINT8 Model Accuracy: "
    f"{accuracy:.2f}% ({correct}/{total} correct)"
)

DEBUG: matmul X1(6, 4) * X2(4, 8)
DEBUG: matmul X1(6, 8) * X2(8, 1)

--- INT8 Model Predictions ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9891 | Predicted: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Predicted: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Predicted: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9960 | Predicted: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0382 | Predicted: Dislike
Food: Opor Ayam              | Prob(Like): 0.9406 | Predicted: Like

INT8 Model Accuracy: 100.00% (6/6 correct)


In [14]:
###########################################################################
# Quantization error calculation
###########################################################################

model_quantized.eval()

with torch.no_grad():
    raw_logits_int8 = model_quantized(X_tensor)
    probabilities_int8 = torch.sigmoid(raw_logits_int8)
    predictions_int8 = (probabilities_int8 >= 0.5).float()

errors = torch.abs(probabilities_fp32 - probabilities_int8)

print("\n--- FP32 vs INT8 Model ---")
print(
    f"{'Food':22s} | "
    f"{'FP32 Prob':>10s} | "
    f"{'INT8 Prob':>10s} | "
    f"{'Abs Error':>10s} | "
    f"{'FP32 Pred':>10s} | "
    f"{'INT8 Pred':>10s}"
)
print("-" * 90)

for name, p_fp32, p_int8, error, pred_fp32, pred_int8 in zip(
    df['Name'],
    probabilities_fp32,
    probabilities_int8,
    errors,
    predictions_fp32,
    predictions_int8
):
    taste_fp32 = "Like" if pred_fp32.item() else "Dislike"
    taste_int8 = "Like" if pred_int8.item() else "Dislike"

    print(
        f"{name:22s} | "
        f"{p_fp32.item():10.4f} | "
        f"{p_int8.item():10.4f} | "
        f"{error.item():10.4f} | "
        f"{taste_fp32:>10s} | "
        f"{taste_int8:>10s}"
    )

print("-" * 90)
print(f"Mean Absolute Error:    {errors.mean().item():.6f}")
print(f"Maximum Absolute Error: {errors.max().item():.6f}")

DEBUG: matmul X1(6, 4) * X2(4, 8)
DEBUG: matmul X1(6, 8) * X2(8, 1)

--- FP32 vs INT8 Model ---
Food                   |  FP32 Prob |  INT8 Prob |  Abs Error |  FP32 Pred |  INT8 Pred
------------------------------------------------------------------------------------------
Nasi Tim Ayam          |     0.9892 |     0.9891 |     0.0001 |       Like |       Like
Telur Balado           |     0.0067 |     0.0067 |     0.0000 |    Dislike |    Dislike
Sayur Bening Bayam     |     0.0003 |     0.0003 |     0.0000 |    Dislike |    Dislike
Ayam Goreng Mentega    |     0.9961 |     0.9960 |     0.0000 |       Like |       Like
Tumis Buncis Wortel    |     0.0387 |     0.0382 |     0.0005 |    Dislike |    Dislike
Opor Ayam              |     0.9408 |     0.9406 |     0.0002 |       Like |       Like
------------------------------------------------------------------------------------------
Mean Absolute Error:    0.000144
Maximum Absolute Error: 0.000493
